# NB11: Transport Reaction Evidence Integration

**Purpose**: Build a transport-specific evidence integration pipeline parallel to the
EC-based enzyme mapping. NB10 showed that 82.5% of transport reactions are unmapped
because they lack EC numbers. This notebook collects evidence from multiple annotation
layers — protein names, GO terms, comment_xml function descriptions, and InterPro
domains — and matches substrate-specific annotations to transport reactions via their
actual reagent molecules.

**Context**: 4,954 of 6,004 balanced transport reactions are unmapped. The reagent-molecule
join gives us 2,180 unique non-cofactor substrate names across 4,800 reactions. Rather than
parsing reaction name text, we match UniProt protein annotations against these molecule names.

**Requires**: BERDL JupyterHub (Spark session for GO, comment_xml, InterPro queries)

**Output**: `transport_evidence_mapping.parquet`, `transport_evidence.png`

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import re
import gc
from collections import defaultdict

DATA_DIR = '../data'
FIG_DIR = '../figures'
USER_DIR = '../user_data'

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 12,
})

## 1. Build Transport Reaction Substrate Index

Join reagents → molecules to get precise substrate names for each transport reaction.
Filter out cofactors (H+, H2O, ATP, etc.) to isolate the transported molecule(s).

In [2]:
rxns = pd.read_csv(f'{DATA_DIR}/reactions_all.tsv', sep='\t')
rxns = rxns[rxns['status'] == 'OK'].copy()
transport_rxns = rxns[rxns['is_transport']].copy()
print(f'Balanced transport reactions: {len(transport_rxns):,}')

evidence = pd.read_parquet(f'{DATA_DIR}/evidence_integration_summary.parquet',
                           columns=['rxn_bare', 'any_evidence'])
transport_rxns['rxn_bare'] = transport_rxns['id'].str.replace('seed.reaction:', '', regex=False)
transport_rxns = transport_rxns.merge(evidence, on='rxn_bare', how='left')
transport_rxns['any_evidence'] = transport_rxns['any_evidence'].fillna(False)

unmapped_transport = transport_rxns[~transport_rxns['any_evidence']]
mapped_transport = transport_rxns[transport_rxns['any_evidence']]
print(f'Mapped: {len(mapped_transport):,}  Unmapped: {len(unmapped_transport):,}')

reagents = pd.read_csv(f'{DATA_DIR}/reagents_all.tsv', sep='\t')
mol = pd.read_csv(f'{DATA_DIR}/molecules_all.tsv', sep='\t')[['id', 'name']].rename(
    columns={'name': 'mol_name'})

COFACTORS = {
    'H+', 'H2O', 'ATP', 'ADP', 'Phosphate', 'NAD', 'NADH', 'NADP', 'NADPH',
    'CoA', 'CO2', 'O2', 'H2O2', 'PPi', 'FAD', 'FADH2', 'Pi', 'GTP', 'GDP',
    'UTP', 'UDP', 'CTP', 'CDP', 'diphosphate', 'Orthophosphate',
}

transport_reagents = reagents[reagents['reaction_id'].isin(set(transport_rxns['id']))].merge(
    mol, left_on='molecule_id', right_on='id', how='left')

rxn_substrates = defaultdict(set)
for _, row in transport_reagents.iterrows():
    mol_name = row['mol_name']
    if pd.notna(mol_name) and mol_name not in COFACTORS:
        rxn_id = row['reaction_id'].replace('seed.reaction:', '')
        rxn_substrates[rxn_id].add(mol_name)

rxns_with_subs = sum(1 for s in rxn_substrates.values() if s)
all_substrates = set()
for s in rxn_substrates.values():
    all_substrates |= s

print(f'\nTransport reactions with non-cofactor substrates: {rxns_with_subs:,} / {len(transport_rxns):,}')
print(f'Unique substrate names: {len(all_substrates):,}')
print(f'\nSample substrates:')
for s in sorted(all_substrates)[:20]:
    print(f'  {s}')

del reagents, mol, transport_reagents
gc.collect()

Balanced transport reactions: 6,004
Mapped: 1,050  Unmapped: 4,954



Transport reactions with non-cofactor substrates: 5,797 / 6,004
Unique substrate names: 2,753

Sample substrates:
  (+)-7-Isojasmonic acid
  (+)-Xylose
  (+-)-cis-1-acetyl-4-(p-((2-(2,4-dichlorophenyl)-2-(imidazol-1- ylmethyl)-1,3-dioxolan-4-yl)methoxy)phenyl)piperazine
  (1,3)-beta-xylobiose
  (1,3)-beta-xylotetraose
  (1,3)-beta-xylotriose
  (1,6)-beta-galactobiose
  (1R)-1,2,10,10a-tetrahydrophenazine-1-carboxylate
  (1R,6R)-1,2,5,5a,6,7-hexahydrophenazine-1,6-dicarboxylate
  (2-{[2,3-bis(octadecanoyloxy)propyl phosphonato]oxy}ethyl)trimethylazanium
  (22R,23R)-22,23-Dihydroxy-campest-4-en-3-one
  (25R)-3-oxo-4-cholesten-26-oate
  (2E)-Decenoyl-CoA
  (2E)-Dodecenoyl-CoA
  (2E)-Hexadecenoyl-CoA
  (2E)-Hexadecenoyl-[acp]
  (2E)-Hexenoyl-CoA
  (2E)-Octenoyl-CoA
  (2E)-Tetradecenoyl-CoA
  (2R,4S)-2-methyl-2,3,3,4-tetrahydroxytetrahydrofuran


20

In [3]:
substrate_to_lower = {s: s.lower() for s in all_substrates}
lower_to_substrates = defaultdict(set)
for s, low in substrate_to_lower.items():
    lower_to_substrates[low].add(s)

CATEGORY_MAP = {
    'amino acid': [
        'L-Alanine', 'L-Arginine', 'L-Asparagine', 'L-Aspartate', 'L-Cysteine',
        'L-Glutamate', 'L-Glutamine', 'Glycine', 'L-Histidine', 'L-Isoleucine',
        'L-Leucine', 'L-Lysine', 'L-Methionine', 'L-Phenylalanine', 'L-Proline',
        'L-Serine', 'L-Threonine', 'L-Tryptophan', 'L-Tyrosine', 'L-Valine',
        'D-Alanine', 'D-Serine', 'D-Glutamate', 'D-Aspartate',
    ],
    'sugar': [
        'D-Glucose', 'D-Fructose', 'D-Galactose', 'D-Mannose', 'D-Xylose',
        'L-Arabinose', 'D-Ribose', 'Sucrose', 'Maltose', 'Lactose', 'Trehalose',
        'D-Glucosamine', 'N-Acetyl-D-glucosamine', 'D-Sorbitol', 'D-Mannitol',
        'myo-Inositol', 'L-Fucose', 'L-Rhamnose', 'D-Glucuronate',
    ],
    'nucleoside': [
        'Adenosine', 'Guanosine', 'Cytidine', 'Uridine', 'Thymidine',
        'Inosine', 'Xanthosine',
    ],
    'nucleotide': [
        'AMP', 'GMP', 'CMP', 'UMP', 'dAMP', 'dGMP', 'dCMP', 'dTMP',
    ],
    'vitamin': [
        'Thiamine', 'Riboflavin', 'Niacin', 'Pyridoxine', 'Biotin',
        'Folate', 'Cobalamin', 'Pantothenate', 'Ascorbate',
    ],
    'metal ion': [
        'Fe2+', 'Fe3+', 'Zn2+', 'Cu2+', 'Mn2+', 'Co2+', 'Ni2+', 'Mg2+',
        'Ca2+', 'K+', 'Na+', 'Cd2+', 'Mo',
    ],
    'organic acid': [
        'Citrate', 'Succinate', 'Fumarate', 'Malate', 'Pyruvate',
        'Acetate', 'Lactate', 'Formate', 'Oxaloacetate', 'Propanoate',
    ],
    'fatty acid': [
        'Palmitate', 'Oleate', 'Stearate', 'Myristate', 'Laurate',
    ],
    'peptide': [
        'Dipeptide', 'Tripeptide', 'Oligopeptide', 'Glutathione',
    ],
    'phospholipid': [
        'Phosphatidylcholine', 'Phosphatidylethanolamine', 'Phosphatidylglycerol',
        'Phosphatidylserine', 'Phosphatidylinositol',
    ],
    'dicarboxylate': [
        'Succinate', 'Fumarate', 'Malate', 'Oxaloacetate', 'alpha-Ketoglutarate',
    ],
    'inorganic ion': [
        'Sulfate', 'Nitrate', 'Nitrite', 'Chloride', 'Chromate', 'Molybdate',
        'Tungstate', 'Selenate', 'Arsenate', 'Borate', 'Silicate',
    ],
}

category_substrates = {}
for cat, exemplars in CATEGORY_MAP.items():
    matched = set()
    for ex in exemplars:
        for sub in all_substrates:
            if ex.lower() in sub.lower() or sub.lower() in ex.lower():
                matched.add(sub)
    category_substrates[cat] = matched

category_to_rxns = defaultdict(set)
for cat, subs in category_substrates.items():
    for rxn_bare, rxn_subs in rxn_substrates.items():
        if rxn_subs & subs:
            category_to_rxns[cat].add(rxn_bare)

print('Substrate category coverage of transport reactions:')
for cat in sorted(category_to_rxns, key=lambda c: len(category_to_rxns[c]), reverse=True):
    rxns_hit = category_to_rxns[cat]
    n_subs = len(category_substrates[cat])
    print(f'  {cat}: {len(rxns_hit):,} reactions, {n_subs} substrates matched')

Substrate category coverage of transport reactions:
  amino acid: 485 reactions, 103 substrates matched
  sugar: 435 reactions, 155 substrates matched
  metal ion: 415 reactions, 87 substrates matched
  organic acid: 370 reactions, 62 substrates matched
  inorganic ion: 147 reactions, 59 substrates matched
  dicarboxylate: 130 reactions, 17 substrates matched
  phospholipid: 129 reactions, 99 substrates matched
  nucleotide: 117 reactions, 32 substrates matched
  nucleoside: 114 reactions, 28 substrates matched
  vitamin: 95 reactions, 35 substrates matched
  fatty acid: 33 reactions, 10 substrates matched
  peptide: 24 reactions, 16 substrates matched


## 2. Protein Name Evidence

Use the `name` column (NOT `description`) from `uniprot_transport_proteins.parquet`
to extract substrate keywords and match against the reaction substrate vocabulary.

In [4]:
tp = pd.read_parquet(f'{DATA_DIR}/uniprot_transport_proteins.parquet')
print(f'Transport protein entries: {len(tp):,}')
print(f'Unique proteins: {tp["protein"].nunique():,}')

protein_names = tp.groupby('protein')['name'].apply(
    lambda x: ' | '.join(x.dropna().unique())).reset_index()
protein_names.columns = ['protein', 'all_names']
print(f'Proteins with aggregated names: {len(protein_names):,}')

del tp
gc.collect()

TRANSPORT_STOP = {
    'transport', 'transporter', 'transported', 'transporting', 'transports',
    'permease', 'symport', 'symporter', 'antiport', 'antiporter',
    'efflux', 'porin', 'import', 'export', 'uptake',
    'protein', 'putative', 'probable', 'predicted', 'uncharacterized',
    'family', 'member', 'like', 'related', 'type', 'subunit', 'component',
    'system', 'complex', 'precursor', 'homolog', 'domain', 'containing',
    'membrane', 'inner', 'outer', 'integral', 'transmembrane',
    'binding', 'substrate', 'solute', 'carrier',
    'abc', 'mfs', 'pts', 'major', 'facilitator', 'superfamily',
    'atp', 'dependent', 'driven', 'proton', 'sodium', 'coupled',
    'the', 'of', 'and', 'a', 'an', 'in', 'to', 'for', 'by', 'with',
    'is', 'at', 'or', 'via', 'from', 'into', 'out', 'no', 'not',
    'uniprot', 'submitted', 'recommended', 'full', 'name',
}

def extract_name_substrates(name_text):
    if pd.isna(name_text):
        return set()
    tokens = re.findall(r'[a-zA-Z0-9][-a-zA-Z0-9+]*', name_text.lower())
    return {t for t in tokens if t not in TRANSPORT_STOP and len(t) > 1}

protein_names['name_tokens'] = protein_names['all_names'].apply(extract_name_substrates)

print(f'\nSample protein name tokens:')
for _, row in protein_names.sample(10, random_state=42).iterrows():
    print(f'  {row["all_names"][:80]}')
    print(f'    tokens: {row["name_tokens"]}')

Transport protein entries: 11,392,424


Unique proteins: 10,440,445


Proteins with aggregated names: 10,440,445



Sample protein name tokens:


  Efflux RND transporter periplasmic adaptor subunit
    tokens: {'periplasmic', 'adaptor', 'rnd'}
  ATP-binding protein of ABC transporter
    tokens: {'atp-binding'}
  Equilibrative nucleoside transporter 1
    tokens: {'nucleoside', 'equilibrative'}
  ABC transporter ATP-binding protein
    tokens: {'atp-binding'}
  Amino acid ABC transporter permease
    tokens: {'acid', 'amino'}
  Potassium transport protein
    tokens: {'potassium'}
  Cell division and transport-associated protein TolA
    tokens: {'transport-associated', 'division', 'cell', 'tola'}
  Ribose transport system permease protein
    tokens: {'ribose'}
  NHLM bacteriocin system ABC transporter ATP-binding protein
    tokens: {'bacteriocin', 'nhlm', 'atp-binding'}
  Transport permease protein
    tokens: set()


In [5]:
substrate_lower_set = {s.lower() for s in all_substrates}

category_keywords = {}
for cat in CATEGORY_MAP:
    cat_tokens = set(cat.lower().split())
    category_keywords[cat] = cat_tokens

extra_category_keywords = {
    'amino acid': {'amino', 'aminoacid'},
    'sugar': {'sugar', 'glucose', 'fructose', 'galactose', 'mannose', 'xylose',
              'arabinose', 'ribose', 'sucrose', 'maltose', 'lactose', 'trehalose',
              'glucosamine', 'sorbitol', 'mannitol', 'inositol', 'fucose', 'rhamnose'},
    'nucleoside': {'nucleoside', 'adenosine', 'guanosine', 'cytidine', 'uridine', 'thymidine'},
    'nucleotide': {'nucleotide'},
    'vitamin': {'vitamin', 'thiamine', 'riboflavin', 'niacin', 'pyridoxine', 'biotin',
                'folate', 'cobalamin', 'pantothenate', 'ascorbate'},
    'metal ion': {'iron', 'zinc', 'copper', 'manganese', 'cobalt', 'nickel',
                  'magnesium', 'calcium', 'potassium', 'cadmium', 'molybdenum',
                  'fe2+', 'fe3+', 'zn2+', 'cu2+', 'mn2+', 'co2+', 'ni2+',
                  'divalent', 'cation', 'metal'},
    'organic acid': {'citrate', 'succinate', 'fumarate', 'malate', 'pyruvate',
                     'acetate', 'lactate', 'formate', 'oxaloacetate', 'propanoate',
                     'carboxylate', 'dicarboxylate', 'tricarboxylate'},
    'fatty acid': {'fatty', 'lipid', 'palmitate', 'oleate', 'stearate', 'acyl'},
    'peptide': {'peptide', 'dipeptide', 'tripeptide', 'oligopeptide', 'glutathione'},
    'phospholipid': {'phospholipid', 'phosphatidyl'},
    'dicarboxylate': {'dicarboxylate', 'c4-dicarboxylate'},
    'inorganic ion': {'sulfate', 'nitrate', 'nitrite', 'chloride', 'chromate',
                      'molybdate', 'tungstate', 'selenate', 'arsenate', 'borate',
                      'phosphate', 'anion', 'oxyanion'},
}

for cat, extra in extra_category_keywords.items():
    category_keywords[cat] = category_keywords.get(cat, set()) | extra

substrate_to_rxns = defaultdict(set)
for rxn_bare, subs in rxn_substrates.items():
    for sub in subs:
        substrate_to_rxns[sub].add(rxn_bare)

token_to_substrates = {}
for sub_lower, sub_originals in lower_to_substrates.items():
    if len(sub_lower) < 4:
        continue
    for orig in sub_originals:
        for token_candidate in sub_lower.split('-'):
            if len(token_candidate) >= 4:
                token_to_substrates.setdefault(token_candidate, set()).add(orig)
        token_to_substrates.setdefault(sub_lower, set()).add(orig)

print(f'Token-to-substrate index: {len(token_to_substrates):,} tokens')

name_rxn_counts = defaultdict(lambda: {'n_proteins': 0, 'match_type': 'category', 'substrates': set()})

n_processed = 0
n_matched = 0
for _, prow in protein_names.iterrows():
    tokens = prow['name_tokens']
    if not tokens:
        n_processed += 1
        continue

    protein_matched = False
    matched_rxns = set()

    for token in tokens:
        if len(token) < 4:
            continue
        if token in token_to_substrates:
            for orig_sub in token_to_substrates[token]:
                for rxn_bare in substrate_to_rxns.get(orig_sub, set()):
                    if rxn_bare not in matched_rxns:
                        matched_rxns.add(rxn_bare)
                        entry = name_rxn_counts[rxn_bare]
                        entry['n_proteins'] += 1
                        entry['match_type'] = 'specific'
                        if len(entry['substrates']) < 10:
                            entry['substrates'].add(orig_sub)
                        protein_matched = True

    for cat, cat_kws in category_keywords.items():
        if tokens & cat_kws:
            for rxn_bare in category_to_rxns.get(cat, set()):
                if rxn_bare not in matched_rxns:
                    matched_rxns.add(rxn_bare)
                    entry = name_rxn_counts[rxn_bare]
                    entry['n_proteins'] += 1
                    if len(entry['substrates']) < 10:
                        entry['substrates'].add(cat)
                    protein_matched = True

    if protein_matched:
        n_matched += 1

    n_processed += 1
    if n_processed % 2_000_000 == 0:
        print(f'  Processed {n_processed:,} proteins, {n_matched:,} matched...')

name_evidence_rxn = pd.DataFrame([
    {'rxn_bare': rxn, 'match_type': d['match_type'],
     'n_proteins': d['n_proteins'],
     'matched_substrates': ', '.join(sorted(list(d['substrates'])[:5]))}
    for rxn, d in name_rxn_counts.items()
])

print(f'\nProtein name evidence (reaction-level):')
print(f'  Reactions covered: {len(name_evidence_rxn):,}')
if len(name_evidence_rxn) > 0:
    n_specific = (name_evidence_rxn['match_type'] == 'specific').sum()
    n_category = (name_evidence_rxn['match_type'] == 'category').sum()
    print(f'  Specific: {n_specific:,}  Category: {n_category:,}')
print(f'  Proteins matched: {n_matched:,} / {n_processed:,}')

unmapped_name_rxns = set(name_evidence_rxn['rxn_bare']) & set(unmapped_transport['rxn_bare'])
print(f'  Unmapped reactions covered: {len(unmapped_name_rxns):,} / {len(unmapped_transport):,}')

if len(name_evidence_rxn) > 0:
    print(f'\nSample matches:')
    for _, row in name_evidence_rxn.head(10).iterrows():
        print(f'  {row["rxn_bare"]}: {row["match_type"]}, {row["n_proteins"]} proteins, subs: {row["matched_substrates"]}')

del protein_names, name_rxn_counts
gc.collect()

Token-to-substrate index: 3,952 tokens


  Processed 2,000,000 proteins, 1,050,170 matched...


  Processed 4,000,000 proteins, 1,967,400 matched...


  Processed 6,000,000 proteins, 2,874,279 matched...


  Processed 8,000,000 proteins, 3,750,460 matched...


  Processed 10,000,000 proteins, 4,484,821 matched...



Protein name evidence (reaction-level):
  Reactions covered: 3,585
  Specific: 2,979  Category: 606
  Proteins matched: 4,700,268 / 10,440,445
  Unmapped reactions covered: 3,056 / 4,954

Sample matches:
  rxn12614: specific, 537251 proteins, subs: Chondroitin 6-sulfate, inorganic ion
  rxn18481: specific, 550542 proteins, subs: 4-Deoxy-beta-D-gluc-4-enuronosyl-(1,3)-N-acetyl-D-galactosamine 6-sulfate, inorganic ion
  rxn56039: specific, 537251 proteins, subs: inorganic ion, taurochenodeoxycholate 3-sulfate
  rxn05153: specific, 537251 proteins, subs: Sulfate, inorganic ion
  rxn05238: specific, 537251 proteins, subs: Sulfate, inorganic ion
  rxn13664: specific, 537315 proteins, subs: Sulfate, Sulfite, inorganic ion
  rxn32975: specific, 537251 proteins, subs: Sulfate, inorganic ion
  rxn34447: specific, 1368134 proteins, subs: Sulfate, inorganic ion, metal ion
  rxn31183: specific, 537251 proteins, subs: Sulfate, inorganic ion
  rxn12811: specific, 1368134 proteins, subs: Sulfate, in

0

## 3. GO Term Evidence

Pull transport-related GO term names from `kescience_interpro.go_mapping`, then query
`refdata_uniprot.identifier` for proteins annotated with those GO IDs.

In [6]:
import sys
sys.path.insert(0, '../../scripts')
from berdl_notebook_utils.setup_spark_session import get_spark_session
import pyarrow as pa
import pyarrow.parquet as pq

spark = get_spark_session()
spark.sql('SET spark.sql.autoBroadcastJoinThreshold = -1')

DataFrame[key: string, value: string]

In [7]:
go_mapping = spark.sql("""
    SELECT go_id, go_name FROM kescience_interpro.go_mapping
    WHERE LOWER(go_name) LIKE '%transport%'
       OR LOWER(go_name) LIKE '%permease%'
       OR LOWER(go_name) LIKE '%symport%'
       OR LOWER(go_name) LIKE '%antiport%'
       OR LOWER(go_name) LIKE '%channel%'
       OR LOWER(go_name) LIKE '%porin%'
""").toPandas()

print(f'Transport-related GO terms: {len(go_mapping):,}')
print(f'\nSample GO terms:')
for _, row in go_mapping.head(20).iterrows():
    print(f'  {row["go_id"]}: {row["go_name"]}')

go_substrate_map = {}
for _, row in go_mapping.iterrows():
    go_name_lower = row['go_name'].lower()
    tokens = set(re.findall(r'[a-zA-Z0-9][-a-zA-Z0-9+]*', go_name_lower))
    tokens -= TRANSPORT_STOP
    tokens -= {'activity', 'transmembrane', 'channel', 'involved'}

    matched_subs = set()
    for token in tokens:
        for sub_lower, sub_originals in lower_to_substrates.items():
            if token in sub_lower or sub_lower in token:
                if len(token) >= 4 and len(sub_lower) >= 4:
                    matched_subs |= sub_originals

    matched_cats = set()
    for cat, cat_kws in category_keywords.items():
        if tokens & cat_kws:
            matched_cats.add(cat)

    if matched_subs or matched_cats:
        go_substrate_map[row['go_id']] = {
            'go_name': row['go_name'],
            'specific_substrates': matched_subs,
            'categories': matched_cats,
        }

print(f'\nGO terms with substrate specificity: {len(go_substrate_map):,}')
for go_id, info in list(go_substrate_map.items())[:10]:
    print(f'  {go_id} ({info["go_name"]})')
    if info['specific_substrates']:
        print(f'    specific: {list(info["specific_substrates"])[:5]}')
    if info['categories']:
        print(f'    categories: {info["categories"]}')

Transport-related GO terms: 2,102

Sample GO terms:
  GO:0015473: fimbrial usher porin activity
  GO:0030321: transepithelial chloride transport
  GO:0022857: transmembrane transporter activity
  GO:0071705: nitrogen compound transport
  GO:0006869: lipid transport
  GO:0015379: potassium:chloride symporter activity
  GO:0006811: monoatomic ion transport
  GO:0022857: transmembrane transporter activity
  GO:0055085: transmembrane transport
  GO:0046933: proton-transporting ATP synthase activity, rotational mechanism
  GO:0045259: proton-transporting ATP synthase complex
  GO:0046907: intracellular transport
  GO:0019646: aerobic electron transport chain
  GO:0006886: intracellular protein transport
  GO:0046740: transport of virus in host, cell to cell
  GO:0090482: vitamin transmembrane transporter activity
  GO:0051180: vitamin transport
  GO:0046961: proton-transporting ATPase activity, rotational mechanism
  GO:1902600: proton transmembrane transport
  GO:0033179: proton-transporti


GO terms with substrate specificity: 367
  GO:0030321 (transepithelial chloride transport)
    specific: ['Benzethonium chloride', 'potassium chloride', 'calcium chloride', 'copper (II) chloride', 'D-Glucosamine Hydrochloride']
    categories: {'inorganic ion'}
  GO:0071705 (nitrogen compound transport)
    specific: ['an organophosphorus compound', 'Inorganic-Compounds', 'an organosulfur compound']
  GO:0006869 (lipid transport)
    specific: ['Lipid', 'cold adapted KDO(2)-lipid (A)', 'D-Abequosyl-D-mannosyl-rhamnosyl-D-galactose-1-diphospholipid', 'a glycerophospholipid', 'phosphoethanolamine KDO(2)-lipid (A)']
    categories: {'fatty acid'}
  GO:0015379 (potassium:chloride symporter activity)
    specific: ['Benzethonium chloride', 'potassium chloride', 'calcium chloride', 'copper (II) chloride', 'D-Glucosamine Hydrochloride']
    categories: {'metal ion', 'inorganic ion'}
  GO:0006811 (monoatomic ion transport)
    categories: {'metal ion', 'inorganic ion'}
  GO:0019646 (aerobic e

In [8]:
if go_substrate_map:
    go_ids_str = ', '.join(f"'{gid.replace('GO:', '')}'"
                           for gid in go_substrate_map.keys())

    go_protein_counts = spark.sql(f"""
        SELECT CONCAT('GO:', xref) AS go_id,
               COUNT(DISTINCT REPLACE(entity_id, 'uniprot:', '')) AS n_proteins
        FROM refdata_uniprot.identifier
        WHERE db = 'GO' AND xref IN ({go_ids_str})
        GROUP BY xref
    """).toPandas()

    go_id_protein_counts = dict(zip(go_protein_counts['go_id'],
                                    go_protein_counts['n_proteins']))
    total_go_proteins = go_protein_counts['n_proteins'].sum()

    print(f'GO IDs with protein annotations: {len(go_protein_counts):,}')
    print(f'Total protein-GO associations: {total_go_proteins:,}')
    print(f'\nTop GO terms by protein count:')
    for _, row in go_protein_counts.nlargest(10, 'n_proteins').iterrows():
        name = go_substrate_map.get(row['go_id'], {}).get('go_name', '?')
        print(f'  {row["go_id"]} ({name}): {row["n_proteins"]:,} proteins')
else:
    go_id_protein_counts = {}
    total_go_proteins = 0
    print('No GO terms with substrate specificity found.')

GO IDs with protein annotations: 364
Total protein-GO associations: 12,357,634

Top GO terms by protein count:
  GO:0015990 (electron transport coupled proton transport): 1,993,891 proteins
  GO:0006123 (mitochondrial electron transport, cytochrome c to oxygen): 1,776,653 proteins
  GO:0042773 (ATP synthesis coupled electron transport): 400,867 proteins
  GO:0006865 (amino acid transport): 326,710 proteins
  GO:0006122 (mitochondrial electron transport, ubiquinol to cytochrome c): 279,358 proteins
  GO:0015833 (peptide transport): 273,350 proteins
  GO:0042910 (xenobiotic transmembrane transporter activity): 254,293 proteins
  GO:0022904 (respiratory electron transport chain): 186,546 proteins
  GO:1904680 (peptide transmembrane transporter activity): 154,846 proteins
  GO:0015344 (siderophore uptake transmembrane transporter activity): 144,140 proteins


In [9]:
go_rxn_data = defaultdict(lambda: {'n_proteins': 0, 'match_type': 'category',
                                    'substrates': set()})

for go_id, info in go_substrate_map.items():
    n_prots = go_id_protein_counts.get(go_id, 0)
    if n_prots == 0:
        continue

    for sub in info['specific_substrates']:
        for rxn_bare in substrate_to_rxns.get(sub, set()):
            entry = go_rxn_data[rxn_bare]
            entry['n_proteins'] += n_prots
            entry['match_type'] = 'specific'
            entry['substrates'].add(sub)

    for cat in info['categories']:
        for rxn_bare in category_to_rxns.get(cat, set()):
            if go_rxn_data[rxn_bare]['match_type'] != 'specific':
                go_rxn_data[rxn_bare]['match_type'] = 'category'
            go_rxn_data[rxn_bare]['n_proteins'] += n_prots
            go_rxn_data[rxn_bare]['substrates'].add(cat)

go_evidence_rxn = pd.DataFrame([
    {'rxn_bare': rxn, 'match_type': d['match_type'],
     'n_proteins': d['n_proteins'],
     'matched_substrates': ', '.join(sorted(list(d['substrates'])[:5]))}
    for rxn, d in go_rxn_data.items()
])

unmapped_transport_ids = set(unmapped_transport['rxn_bare'])

print(f'GO term evidence (reaction-level):')
print(f'  Reactions covered: {len(go_evidence_rxn):,}')
if len(go_evidence_rxn) > 0:
    n_specific = (go_evidence_rxn['match_type'] == 'specific').sum()
    n_category = (go_evidence_rxn['match_type'] == 'category').sum()
    print(f'  Specific: {n_specific:,}  Category: {n_category:,}')
    unmapped_go = set(go_evidence_rxn['rxn_bare']) & unmapped_transport_ids
    print(f'  Unmapped reactions covered: {len(unmapped_go):,} / {len(unmapped_transport):,}')
    print(f'  (protein counts are upper bounds — multi-GO proteins counted per term)')
    print(f'\nSample matches:')
    for _, row in go_evidence_rxn.head(10).iterrows():
        print(f'  {row["rxn_bare"]}: {row["match_type"]}, '
              f'~{row["n_proteins"]:,} proteins, subs: {row["matched_substrates"]}')

del go_rxn_data
gc.collect()

GO term evidence (reaction-level):
  Reactions covered: 3,297
  Specific: 2,454  Category: 843
  Unmapped reactions covered: 2,765 / 4,954
  (protein counts are upper bounds — multi-GO proteins counted per term)

Sample matches:
  rxn39193: specific, ~2,713,429 proteins, subs: Benzethonium chloride, inorganic ion
  rxn39178: specific, ~3,248,608 proteins, subs: inorganic ion, potassium chloride
  rxn39176: specific, ~3,054,319 proteins, subs: calcium chloride, inorganic ion
  rxn39115: specific, ~2,756,214 proteins, subs: copper (II) chloride, inorganic ion
  rxn39240: specific, ~2,820,984 proteins, subs: D-Glucosamine Hydrochloride, inorganic ion, sugar
  rxn39204: specific, ~2,713,429 proteins, subs: Sodium hypochloride, inorganic ion
  rxn45356: specific, ~3,370,909 proteins, subs: 10-methyl-3,6-diaminoacridinium chloride, inorganic ion
  rxn60754: specific, ~3,370,909 proteins, subs: 10-methyl-3,6-diaminoacridinium chloride, inorganic ion
  rxn39186: specific, ~6,438,162 proteins, 

0

## 4. comment_xml Function Evidence

Query `refdata_uniprot.comment_xml` for transport-related function descriptions.
Parse XML to extract substrate mentions from function text.

In [10]:
comment_df = spark.sql("""
    SELECT REPLACE(entity_id, 'uniprot:', '') AS protein, content
    FROM refdata_uniprot.comment_xml
    WHERE content LIKE '%transport%'
       OR content LIKE '%permease%'
       OR content LIKE '%symport%'
       OR content LIKE '%antiport%'
""").toPandas()

print(f'Transport-related comment_xml entries: {len(comment_df):,}')
print(f'Unique proteins: {comment_df["protein"].nunique():,}')

print(f'\nSample XML content (first 3):')
for _, row in comment_df.head(3).iterrows():
    print(f'  Protein: {row["protein"]}')
    print(f'  Content: {row["content"][:300]}')
    print()

Transport-related comment_xml entries: 46,681
Unique proteins: 33,324

Sample XML content (first 3):
  Protein: Q8NQC8
  Content: <comment type="function"><text evidence="2 3">Catalyzes the proton motive force-dependent arsenite efflux from the cell. Probably functions as an arsenite/H(+) antiporter. Does not transport antimonite.</text></comment>

  Protein: D2TV88
  Content: <comment type="function"><text evidence="6">Autotransporter required for the colonization of the mouse host gastrointestinal tract, possibly by mediating bacteria adhesion to host cells.</text></comment>

  Protein: D2TV88
  Content: <comment type="domain"><text evidence="1">The signal peptide, cleaved at the inner membrane, guides the autotransporter protein to the periplasmic space (By similarity). Insertion of the C-terminal translocator domain (beta domain) in the outer membrane forms a hydrophilic pore for translocation of 



In [11]:
function_pat = re.compile(r'<text[^>]*>(.*?)</text>', re.DOTALL)

def extract_function_text(xml_content):
    """Extract readable text from comment XML."""
    texts = function_pat.findall(xml_content)
    return ' '.join(texts)

comment_df['function_text'] = comment_df['content'].apply(extract_function_text)

print(f'Entries with function text: {comment_df["function_text"].str.len().gt(0).sum():,}')
print(f'\nSample function texts:')
for _, row in comment_df[comment_df['function_text'].str.len() > 10].head(10).iterrows():
    print(f'  {row["protein"]}: {row["function_text"][:150]}')

Entries with function text: 46,606

Sample function texts:
  Q8NQC8: Catalyzes the proton motive force-dependent arsenite efflux from the cell. Probably functions as an arsenite/H(+) antiporter. Does not transport antim
  D2TV88: Autotransporter required for the colonization of the mouse host gastrointestinal tract, possibly by mediating bacteria adhesion to host cells.
  D2TV88: The signal peptide, cleaved at the inner membrane, guides the autotransporter protein to the periplasmic space (By similarity). Insertion of the C-ter
  P0AE07: Periplasmic adaptor component of the AcrAB-TolC efflux system that confers multidrug resistance. The AcrAB-TolC efflux system has a broad substrate sp
  P0AE06: Periplasmic adaptor component of the AcrAB-TolC efflux system that confers multidrug resistance (PubMed:9878415). The AcrAB-TolC efflux system has a b
  O18935: Interacts with RAB26; RAB26 mediates cell surface transport from the Golgi. Interacts with PPP1R9B. Interacts with GGA1, GGA2 and GGA3

In [12]:
comment_rxn_data = defaultdict(lambda: {'proteins': set(), 'match_type': 'category',
                                         'substrates': set()})

n_with_text = 0
for _, row in comment_df.iterrows():
    protein = row['protein']
    text = row['function_text'].lower()
    if not text:
        continue
    n_with_text += 1

    tokens = set(re.findall(r'[a-zA-Z0-9][-a-zA-Z0-9+]*', text))
    tokens -= TRANSPORT_STOP

    matched_rxns = set()

    for token in tokens:
        if len(token) < 4:
            continue
        if token in token_to_substrates:
            for orig_sub in token_to_substrates[token]:
                for rxn_bare in substrate_to_rxns.get(orig_sub, set()):
                    if rxn_bare not in matched_rxns:
                        entry = comment_rxn_data[rxn_bare]
                        entry['proteins'].add(protein)
                        entry['match_type'] = 'specific'
                        entry['substrates'].add(orig_sub)
                        matched_rxns.add(rxn_bare)

    for cat, cat_kws in category_keywords.items():
        if tokens & cat_kws:
            for rxn_bare in category_to_rxns.get(cat, set()):
                if rxn_bare not in matched_rxns:
                    entry = comment_rxn_data[rxn_bare]
                    entry['proteins'].add(protein)
                    entry['substrates'].add(cat)

comment_evidence_rxn = pd.DataFrame([
    {'rxn_bare': rxn, 'match_type': d['match_type'],
     'n_proteins': len(d['proteins']),
     'matched_substrates': ', '.join(sorted(list(d['substrates'])[:5]))}
    for rxn, d in comment_rxn_data.items()
])

print(f'comment_xml evidence (reaction-level):')
print(f'  Entries with function text: {n_with_text:,}')
print(f'  Reactions covered: {len(comment_evidence_rxn):,}')
if len(comment_evidence_rxn) > 0:
    n_specific = (comment_evidence_rxn['match_type'] == 'specific').sum()
    n_category = (comment_evidence_rxn['match_type'] == 'category').sum()
    print(f'  Specific: {n_specific:,}  Category: {n_category:,}')
    unmapped_comment = set(comment_evidence_rxn['rxn_bare']) & set(unmapped_transport['rxn_bare'])
    print(f'  Unmapped reactions covered: {len(unmapped_comment):,} / {len(unmapped_transport):,}')

del comment_df, comment_rxn_data
gc.collect()

comment_xml evidence (reaction-level):
  Entries with function text: 46,606
  Reactions covered: 3,808
  Specific: 3,330  Category: 478
  Unmapped reactions covered: 3,211 / 4,954


43

## 5. InterPro Domain Evidence

Find transport-related InterPro entries, then pull proteins with those domains
and match domain-derived substrate categories to transport reactions.

In [13]:
ipr_entries = spark.sql("""
    SELECT ipr_id, entry_name FROM kescience_interpro.entry
    WHERE LOWER(entry_name) LIKE '%transport%'
       OR LOWER(entry_name) LIKE '%permease%'
       OR LOWER(entry_name) LIKE '%symport%'
       OR LOWER(entry_name) LIKE '%antiport%'
       OR LOWER(entry_name) LIKE '%porin%'
       OR LOWER(entry_name) LIKE '%channel%'
       OR LOWER(entry_name) LIKE '%efflux%'
""").toPandas()

print(f'Transport-related InterPro entries: {len(ipr_entries):,}')
print(f'\nSample entries:')
for _, row in ipr_entries.head(20).iterrows():
    print(f'  {row["ipr_id"]}: {row["entry_name"]}')

ipr_substrate_map = {}
for _, row in ipr_entries.iterrows():
    name_lower = row['entry_name'].lower()
    tokens = set(re.findall(r'[a-zA-Z0-9][-a-zA-Z0-9+]*', name_lower))
    tokens -= TRANSPORT_STOP
    tokens -= {'superfamily', 'like', 'related', 'fold', 'domain'}

    matched_subs = set()
    for token in tokens:
        for sub_lower, sub_originals in lower_to_substrates.items():
            if token in sub_lower or sub_lower in token:
                if len(token) >= 4 and len(sub_lower) >= 4:
                    matched_subs |= sub_originals

    matched_cats = set()
    for cat, cat_kws in category_keywords.items():
        if tokens & cat_kws:
            matched_cats.add(cat)

    if matched_subs or matched_cats:
        ipr_substrate_map[row['ipr_id']] = {
            'entry_name': row['entry_name'],
            'specific_substrates': matched_subs,
            'categories': matched_cats,
        }

print(f'\nInterPro entries with substrate specificity: {len(ipr_substrate_map):,}')
for ipr_id, info in list(ipr_substrate_map.items())[:10]:
    print(f'  {ipr_id} ({info["entry_name"]})')
    if info['specific_substrates']:
        print(f'    specific: {list(info["specific_substrates"])[:5]}')
    if info['categories']:
        print(f'    categories: {info["categories"]}')

Transport-related InterPro entries: 1,422

Sample entries:
  IPR004840: Amino acid permease, conserved site
  IPR005829: Sugar transporter, conserved site
  IPR006686: Mechanosensitive ion channel MscS, conserved site
  IPR013061: Tryptophan/tryrosine permease, conserved site
  IPR013793: Porin, Gram-negative type, conserved site
  IPR017871: ABC transporter-like, conserved site
  IPR018000: Neurotransmitter-gated ion-channel, conserved site
  IPR018043: Sodium:galactoside symporter, conserved site
  IPR018045: Sulphate anion transporter, conserved site
  IPR018047: Ammonium transporter, conserved site
  IPR018093: BCCT transporter, conserved site
  IPR018107: Sodium:dicarboxylate symporter, conserved site
  IPR018212: Sodium/solute symporter, conserved site
  IPR018456: PTR2 family proton/oligopeptide symporter, conserved site
  IPR018457: LacY/RafB permease family, conserved site
  IPR018892: Retro-transposon transporting motif
  IPR019823: Large-conductance mechanosensitive channel,


InterPro entries with substrate specificity: 915
  IPR004840 (Amino acid permease, conserved site)
    specific: ['3-[[(2S)-2,4-dihydroxy-3,3-dimethylbutanoyl]amino]propanoic acid', 'Docosahexaenoic acid', "2'-Hydroxyferulic acid", 'N-(4-aminobenzoyl)-L-glutamate', 'Cobalt (III) Ethylenediaminetetraacetic acid']
    categories: {'fatty acid', 'amino acid', 'organic acid'}
  IPR005829 (Sugar transporter, conserved site)
    specific: ['Sugar-alcohols', 'Sugar-Phosphate', 'Sugar', 'a nucleotide sugar', 'NUCLEOTIDE-SUGARS']
    categories: {'sugar'}
  IPR006686 (Mechanosensitive ion channel MscS, conserved site)
    categories: {'metal ion', 'inorganic ion'}
  IPR013061 (Tryptophan/tryrosine permease, conserved site)
    specific: ['L-Tryptophan', '4-fluoro-L-tryptophan']
  IPR018043 (Sodium:galactoside symporter, conserved site)
    specific: ['Methyl beta-D-galactoside', 'LACT']
  IPR018045 (Sulphate anion transporter, conserved site)
    specific: ['Anions']
    categories: {'inorgani

In [14]:
if ipr_substrate_map:
    ipr_ids_str = ', '.join(f"'{iid}'" for iid in ipr_substrate_map.keys())

    ipr_protein_counts = spark.sql(f"""
        SELECT ipr_id,
               COUNT(DISTINCT uniprot_acc) AS n_proteins
        FROM kescience_interpro.protein2ipr
        WHERE ipr_id IN ({ipr_ids_str})
        GROUP BY ipr_id
    """).toPandas()

    ipr_id_protein_counts = dict(zip(ipr_protein_counts['ipr_id'],
                                     ipr_protein_counts['n_proteins']))
    total_ipr_proteins = ipr_protein_counts['n_proteins'].sum()

    print(f'IPR IDs with protein annotations: {len(ipr_protein_counts):,}')
    print(f'Total protein-IPR associations: {total_ipr_proteins:,}')
    print(f'\nTop InterPro entries by protein count:')
    for _, row in ipr_protein_counts.nlargest(10, 'n_proteins').iterrows():
        name = ipr_substrate_map.get(row['ipr_id'], {}).get('entry_name', '?')
        print(f'  {row["ipr_id"]} ({name}): {row["n_proteins"]:,} proteins')
else:
    ipr_id_protein_counts = {}
    total_ipr_proteins = 0
    print('No InterPro entries with substrate specificity found.')

IPR IDs with protein annotations: 915
Total protein-IPR associations: 13,900,958

Top InterPro entries by protein count:
  IPR037185 (Multidrug transporter EmrE superfamily): 458,667 proteins
  IPR001750 (NADH:quinone oxidoreductase/Mrp antiporter, transmembrane domain): 380,562 proteins
  IPR005829 (Sugar transporter, conserved site): 375,703 proteins
  IPR005828 (Major facilitator, sugar transporter-like): 342,381 proteins
  IPR005821 (Ion transport domain): 241,867 proteins
  IPR003838 (ABC3 transporter permease, C-terminal): 205,484 proteins
  IPR002293 (Amino acid/polyamine transporter I): 188,809 proteins
  IPR003663 (Sugar/inositol transporter): 173,293 proteins
  IPR013563 (Oligopeptide/dipeptide ABC transporter, C-terminal): 162,544 proteins
  IPR027463 (Multidrug efflux transporter AcrB TolC docking domain, DN/DC subdomains): 140,769 proteins


In [15]:
ipr_rxn_data = defaultdict(lambda: {'n_proteins': 0, 'match_type': 'category',
                                     'substrates': set()})

for ipr_id, info in ipr_substrate_map.items():
    n_prots = ipr_id_protein_counts.get(ipr_id, 0)
    if n_prots == 0:
        continue

    for sub in info['specific_substrates']:
        for rxn_bare in substrate_to_rxns.get(sub, set()):
            entry = ipr_rxn_data[rxn_bare]
            entry['n_proteins'] += n_prots
            entry['match_type'] = 'specific'
            entry['substrates'].add(sub)

    for cat in info['categories']:
        for rxn_bare in category_to_rxns.get(cat, set()):
            if ipr_rxn_data[rxn_bare]['match_type'] != 'specific':
                ipr_rxn_data[rxn_bare]['match_type'] = 'category'
            ipr_rxn_data[rxn_bare]['n_proteins'] += n_prots
            ipr_rxn_data[rxn_bare]['substrates'].add(cat)

ipr_evidence_rxn = pd.DataFrame([
    {'rxn_bare': rxn, 'match_type': d['match_type'],
     'n_proteins': d['n_proteins'],
     'matched_substrates': ', '.join(sorted(list(d['substrates'])[:5]))}
    for rxn, d in ipr_rxn_data.items()
])

print(f'InterPro evidence (reaction-level):')
print(f'  Reactions covered: {len(ipr_evidence_rxn):,}')
if len(ipr_evidence_rxn) > 0:
    n_specific = (ipr_evidence_rxn['match_type'] == 'specific').sum()
    n_category = (ipr_evidence_rxn['match_type'] == 'category').sum()
    print(f'  Specific: {n_specific:,}  Category: {n_category:,}')
    unmapped_ipr = set(ipr_evidence_rxn['rxn_bare']) & set(unmapped_transport['rxn_bare'])
    print(f'  Unmapped reactions covered: {len(unmapped_ipr):,} / {len(unmapped_transport):,}')
    print(f'  (protein counts are upper bounds — multi-domain proteins counted per entry)')
    print(f'\nSample matches:')
    for _, row in ipr_evidence_rxn.head(10).iterrows():
        print(f'  {row["rxn_bare"]}: {row["match_type"]}, '
              f'~{row["n_proteins"]:,} proteins, subs: {row["matched_substrates"]}')

del ipr_rxn_data
gc.collect()

InterPro evidence (reaction-level):
  Reactions covered: 3,878
  Specific: 3,379  Category: 499
  Unmapped reactions covered: 3,246 / 4,954
  (protein counts are upper bounds — multi-domain proteins counted per entry)

Sample matches:
  rxn39130: specific, ~1,634,175 proteins, subs: 3-[[(2S)-2,4-dihydroxy-3,3-dimethylbutanoyl]amino]propanoic acid
  rxn56377: specific, ~1,623,724 proteins, subs: Docosahexaenoic acid
  rxn33107: specific, ~1,623,724 proteins, subs: 2'-Hydroxyferulic acid
  rxn49838: specific, ~3,152,591 proteins, subs: N-(4-aminobenzoyl)-L-glutamate, amino acid
  rxn50337: specific, ~3,152,591 proteins, subs: N-(4-aminobenzoyl)-L-glutamate, amino acid
  rxn39135: specific, ~1,700,632 proteins, subs: Cobalt (III) Ethylenediaminetetraacetic acid
  rxn18576: specific, ~1,691,596 proteins, subs: Polar-amino-acids
  rxn56160: specific, ~1,623,724 proteins, subs: Kainic acid
  rxn05475: specific, ~1,632,625 proteins, subs: 3-aminobutanoic acid
  rxn29740: specific, ~3,082,999 

0

## 6. Evidence Integration & Confidence Scoring

Merge all evidence channels and assign confidence based on evidence depth
and substrate specificity.

In [16]:
unmapped_rxn_ids = set(unmapped_transport['rxn_bare'])

evidence_rxns = {
    'name': name_evidence_rxn if len(name_evidence_rxn) > 0 else pd.DataFrame(columns=['rxn_bare', 'match_type', 'n_proteins']),
    'go': go_evidence_rxn if len(go_evidence_rxn) > 0 else pd.DataFrame(columns=['rxn_bare', 'match_type', 'n_proteins']),
    'comment': comment_evidence_rxn if len(comment_evidence_rxn) > 0 else pd.DataFrame(columns=['rxn_bare', 'match_type', 'n_proteins']),
    'interpro': ipr_evidence_rxn if len(ipr_evidence_rxn) > 0 else pd.DataFrame(columns=['rxn_bare', 'match_type', 'n_proteins']),
}

print('Per-layer reaction coverage (all transport reactions):')
for layer_name, layer_df in evidence_rxns.items():
    if len(layer_df) > 0:
        n_rxns = len(layer_df)
        unmapped_covered = len(set(layer_df['rxn_bare']) & unmapped_rxn_ids)
        total_prots = layer_df['n_proteins'].sum()
        print(f'  {layer_name}: {n_rxns:,} reactions ({unmapped_covered:,} unmapped), ~{total_prots:,} protein annotations')
    else:
        print(f'  {layer_name}: 0 reactions')

all_evidence = []
for layer_name, layer_df in evidence_rxns.items():
    if len(layer_df) == 0:
        continue
    subset = layer_df[['rxn_bare', 'match_type', 'n_proteins']].copy()
    subset = subset.rename(columns={
        'match_type': f'type_{layer_name}',
        'n_proteins': f'n_proteins_{layer_name}',
    })
    subset[f'has_{layer_name}'] = True
    all_evidence.append(subset)

if all_evidence:
    combined = all_evidence[0]
    for df in all_evidence[1:]:
        combined = combined.merge(df, on='rxn_bare', how='outer')

    layer_cols = [c for c in combined.columns if c.startswith('has_')]
    for col in layer_cols:
        combined[col] = combined[col].fillna(False)

    prot_cols = [c for c in combined.columns if c.startswith('n_proteins_')]
    for col in prot_cols:
        combined[col] = combined[col].fillna(0).astype(int)

    combined['n_layers'] = combined[layer_cols].sum(axis=1).astype(int)

    type_cols = [c for c in combined.columns if c.startswith('type_')]
    combined['has_specific'] = False
    for col in type_cols:
        combined['has_specific'] = combined['has_specific'] | (combined[col] == 'specific')

    combined['total_proteins'] = combined[prot_cols].sum(axis=1)

    def assign_confidence(row):
        if row['n_layers'] >= 3 and row['has_specific']:
            return 'high'
        elif row['n_layers'] >= 2 or (row['n_layers'] == 1 and row['has_specific']):
            return 'medium'
        elif row['n_layers'] == 1:
            return 'low'
        return 'none'

    combined['confidence'] = combined.apply(assign_confidence, axis=1)

    print(f'\nCombined evidence (reaction-level):')
    print(f'  Total reactions with any evidence: {len(combined):,}')

    print(f'\nConfidence distribution:')
    for conf in ['high', 'medium', 'low']:
        n = (combined['confidence'] == conf).sum()
        print(f'  {conf}: {n:,} reactions')

    print(f'\nEvidence depth distribution:')
    for n_lay in sorted(combined['n_layers'].unique()):
        n = (combined['n_layers'] == n_lay).sum()
        print(f'  {int(n_lay)} layer(s): {n:,} reactions')
else:
    combined = pd.DataFrame(columns=['rxn_bare', 'n_layers', 'confidence'])
    print('No evidence from any layer.')

Per-layer reaction coverage (all transport reactions):
  name: 3,585 reactions (3,056 unmapped), ~1,595,324,464 protein annotations
  go: 3,297 reactions (2,765 unmapped), ~3,029,068,160 protein annotations
  comment: 3,808 reactions (3,211 unmapped), ~8,175,622 protein annotations
  interpro: 3,878 reactions (3,246 unmapped), ~5,021,916,438 protein annotations

Combined evidence (reaction-level):
  Total reactions with any evidence: 4,627

Confidence distribution:
  high: 3,140 reactions
  medium: 1,487 reactions
  low: 0 reactions

Evidence depth distribution:
  1 layer(s): 598 reactions
  2 layer(s): 714 reactions
  3 layer(s): 718 reactions
  4 layer(s): 2,597 reactions


/tmp/ipykernel_59555/1364000652.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  combined[col] = combined[col].fillna(False)


In [17]:
if len(combined) > 0:
    rxns_with_transport_evidence = set(combined['rxn_bare'])
    newly_covered = rxns_with_transport_evidence & unmapped_rxn_ids
    already_covered = rxns_with_transport_evidence & set(mapped_transport['rxn_bare'])
    still_unmapped = unmapped_rxn_ids - rxns_with_transport_evidence

    print(f'Transport reaction coverage update:')
    print(f'  Previously mapped (via EC): {len(mapped_transport):,}')
    print(f'  Newly covered by transport evidence: {len(newly_covered):,}')
    print(f'  Already-mapped also recovered: {len(already_covered):,}')
    print(f'  Still unmapped: {len(still_unmapped):,} / {len(transport_rxns):,}')
    print(f'  New coverage: {100*(len(mapped_transport) + len(newly_covered))/len(transport_rxns):.1f}%'
          f' (was {100*len(mapped_transport)/len(transport_rxns):.1f}%)')

    newly_best = combined[combined['rxn_bare'].isin(newly_covered)]
    print(f'\nConfidence of newly-covered reactions:')
    print(f'  High: {(newly_best["confidence"]=="high").sum():,}')
    print(f'  Medium: {(newly_best["confidence"]=="medium").sum():,}')
    print(f'  Low: {(newly_best["confidence"]=="low").sum():,}')
else:
    print('No transport evidence generated.')

Transport reaction coverage update:
  Previously mapped (via EC): 1,050
  Newly covered by transport evidence: 3,903
  Already-mapped also recovered: 724
  Still unmapped: 1,051 / 6,004
  New coverage: 82.5% (was 17.5%)

Confidence of newly-covered reactions:
  High: 2,622
  Medium: 1,281
  Low: 0


In [18]:
if len(combined) > 0:
    save_cols = ['rxn_bare', 'n_layers', 'has_specific', 'confidence', 'total_proteins']
    save_cols += [c for c in combined.columns if c.startswith('has_') and c != 'has_specific']
    save_cols += [c for c in combined.columns if c.startswith('n_proteins_')]
    save_df = combined[save_cols].copy()

    table = pa.table({col: save_df[col].tolist() for col in save_df.columns})
    pq.write_table(table, f'{DATA_DIR}/transport_evidence_mapping.parquet')
    print(f'Saved: transport_evidence_mapping.parquet ({len(save_df):,} rows)')
    del table
else:
    print('No data to save.')

Saved: transport_evidence_mapping.parquet (4,627 rows)


## 7. Summary & Figures

In [19]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
layer_names_fig = []
layer_counts_fig = []
for layer_name, layer_df in evidence_rxns.items():
    if len(layer_df) > 0:
        n_unmapped = len(set(layer_df['rxn_bare']) & unmapped_rxn_ids)
        layer_names_fig.append(layer_name)
        layer_counts_fig.append(n_unmapped)

if layer_names_fig:
    y_pos = range(len(layer_names_fig))
    ax.barh(y_pos, layer_counts_fig,
            color=['#4c78a8', '#f58518', '#e45756', '#72b7b2'][:len(layer_names_fig)])
    ax.set_yticks(y_pos)
    ax.set_yticklabels([n.capitalize() for n in layer_names_fig])
    ax.set_xlabel('Unmapped Transport Reactions Covered')
    ax.set_title('Per-Layer Coverage')
    for i, v in enumerate(layer_counts_fig):
        ax.text(v + 20, i, str(v), va='center')
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
if len(combined) > 0:
    newly_best = combined[combined['rxn_bare'].isin(unmapped_rxn_ids)]

    conf_order = ['high', 'medium', 'low']
    conf_colors = {'high': '#4c78a8', 'medium': '#f58518', 'low': '#e45756'}
    conf_counts = [len(newly_best[newly_best['confidence'] == c]) for c in conf_order]
    still_unmapped_count = len(unmapped_rxn_ids) - sum(conf_counts)

    labels = conf_order + ['still unmapped']
    sizes = conf_counts + [still_unmapped_count]
    colors = [conf_colors[c] for c in conf_order] + ['#bab0ac']

    ax.bar(range(len(labels)), sizes, color=colors)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels([l.capitalize() for l in labels], rotation=30, ha='right')
    ax.set_ylabel('Transport Reactions')
    ax.set_title('Unmapped Transport: Evidence Confidence')
    for i, v in enumerate(sizes):
        ax.text(i, v + 20, str(v), ha='center', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('NB11: Transport Reaction Evidence Integration', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/transport_evidence.png')
plt.show()
print('Saved: transport_evidence.png')

Saved: transport_evidence.png


In [20]:
print('=' * 65)
print('NB11 TRANSPORT EVIDENCE INTEGRATION SUMMARY')
print('=' * 65)

print(f'\n--- Transport Reaction Substrate Index ---')
print(f'Transport reactions (balanced): {len(transport_rxns):,}')
print(f'With non-cofactor substrates: {sum(1 for s in rxn_substrates.values() if s):,}')
print(f'Unique substrate names: {len(all_substrates):,}')

print(f'\n--- Per-Layer Evidence (reaction-level) ---')
for layer_name, layer_df in evidence_rxns.items():
    if len(layer_df) > 0:
        n_rxns = len(layer_df)
        n_unmapped = len(set(layer_df['rxn_bare']) & unmapped_rxn_ids)
        n_specific = (layer_df['match_type'] == 'specific').sum() if 'match_type' in layer_df.columns else 0
        n_category = (layer_df['match_type'] == 'category').sum() if 'match_type' in layer_df.columns else 0
        total_prots = layer_df['n_proteins'].sum()
        print(f'  {layer_name}: {n_rxns:,} rxns ({n_unmapped:,} unmapped), '
              f'~{total_prots:,} protein annotations, '
              f'{n_specific:,} specific / {n_category:,} category')
    else:
        print(f'  {layer_name}: no matches')

if len(combined) > 0:
    rxns_with_evidence = set(combined['rxn_bare'])
    newly_covered = rxns_with_evidence & unmapped_rxn_ids
    still_uncovered = unmapped_rxn_ids - rxns_with_evidence

    print(f'\n--- Coverage Impact ---')
    print(f'Previously mapped (EC-based): {len(mapped_transport):,} / {len(transport_rxns):,} '
          f'({100*len(mapped_transport)/len(transport_rxns):.1f}%)')
    print(f'Newly covered by transport evidence: {len(newly_covered):,}')
    total_now = len(mapped_transport) + len(newly_covered)
    print(f'Total transport coverage: {total_now:,} / {len(transport_rxns):,} '
          f'({100*total_now/len(transport_rxns):.1f}%)')
    print(f'Remaining unmapped: {len(still_uncovered):,} '
          f'({100*len(still_uncovered)/len(transport_rxns):.1f}%)')

    print(f'\n--- Confidence Distribution ---')
    for conf in ['high', 'medium', 'low']:
        n = (combined['confidence'] == conf).sum()
        print(f'  {conf}: {n:,} reactions')

    print(f'\n--- Evidence Depth ---')
    for n_lay in sorted(combined['n_layers'].unique()):
        n = (combined['n_layers'] == n_lay).sum()
        print(f'  {int(n_lay)} layer(s): {n:,} reactions')
else:
    print(f'\nNo transport evidence generated.')

gc.collect()

NB11 TRANSPORT EVIDENCE INTEGRATION SUMMARY

--- Transport Reaction Substrate Index ---
Transport reactions (balanced): 6,004
With non-cofactor substrates: 5,797
Unique substrate names: 2,753

--- Per-Layer Evidence (reaction-level) ---
  name: 3,585 rxns (3,056 unmapped), ~1,595,324,464 protein annotations, 2,979 specific / 606 category
  go: 3,297 rxns (2,765 unmapped), ~3,029,068,160 protein annotations, 2,454 specific / 843 category
  comment: 3,808 rxns (3,211 unmapped), ~8,175,622 protein annotations, 3,330 specific / 478 category
  interpro: 3,878 rxns (3,246 unmapped), ~5,021,916,438 protein annotations, 3,379 specific / 499 category

--- Coverage Impact ---
Previously mapped (EC-based): 1,050 / 6,004 (17.5%)
Newly covered by transport evidence: 3,903
Total transport coverage: 4,953 / 6,004 (82.5%)
Remaining unmapped: 1,051 (17.5%)

--- Confidence Distribution ---
  high: 3,140 reactions
  medium: 1,487 reactions
  low: 0 reactions

--- Evidence Depth ---
  1 layer(s): 598 reac

15